In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# ---------------------------------------------------------
# 1. Simulate the Data (Supply-Demand Imbalances)
# ---------------------------------------------------------
np.random.seed(42)
n_markets = 500

# Simulating high-dimensional features for various housing markets
data = {
    'Median_Home_Price': np.random.lognormal(mean=12.5, sigma=0.8, size=n_markets), # Right-skewed prices
    'Inventory_Growth_Rate': np.random.normal(loc=-0.02, scale=0.05, size=n_markets), # Fractional percentages
    'Days_on_Market': np.random.poisson(lam=45, size=n_markets),
    'New_Permits_Issued': np.random.poisson(lam=150, size=n_markets),
    'Interest_Rate_Local_Variance': np.random.normal(loc=0.0, scale=0.001, size=n_markets), # Low-variance noise
    'Zoning_Strictness_Index': np.random.uniform(low=1, high=10, size=n_markets)
}

df_housing = pd.DataFrame(data)

# ---------------------------------------------------------
# 2. Structural Scaling
# ---------------------------------------------------------
# We use RobustScaler because 'Median_Home_Price' contains heavy right-tail outliers
# which would corrupt a standard Z-score mean/variance calculation.
scaler = RobustScaler()
X_scaled = scaler.fit_transform(df_housing)

# ---------------------------------------------------------
# 3. PCA: Linear Denoising & Eigendecomposition
# ---------------------------------------------------------
# We initialize PCA to keep enough components to explain 90% of the variance
pca = PCA(n_components=0.90, random_state=42)
X_pca = pca.fit_transform(X_scaled)

print(f"Original Dimensions: {X_scaled.shape[1]}")
print(f"Reduced Dimensions (PCA): {X_pca.shape[1]}\n")

# Extracting the Eigenvalues (Explained Variance)
# This shows exactly how much "signal" each new orthogonal axis captured
variance_ratios = pca.explained_variance_ratio_

for i, ratio in enumerate(variance_ratios):
    print(f"Principal Component {i+1} captures {ratio:.2%} of the total variance.")
    
print(f"\nTotal Variance Retained: {np.sum(variance_ratios):.2%}")

# ---------------------------------------------------------
# 4. t-SNE: Nonlinear Local Topology Mapping
# ---------------------------------------------------------
# We feed the denoised PCA output (X_pca) directly into t-SNE, avoiding the 
# high-dimensional distance computation bottleneck.
tsne = TSNE(
    n_components=2,          # Compress down to 2D for visualization
    perplexity=30,           # The expected size of local neighborhoods
    learning_rate='auto',    # Gradient descent step size
    init='pca',              # A more stable initialization than random
    random_state=42
)

# X_tsne contains the final 2D coordinates ready for a scatter plot
X_tsne = tsne.fit_transform(X_pca)

print(f"\nt-SNE Mapping Complete. Final Shape: {X_tsne.shape}")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import KBinsDiscretizer

# 1. Simulate the Skewed Dataset 
# (Income represented in thousands: $40k to $90k, plus one $5M outlier)
income = np.array([40, 45, 50, 55, 60, 65, 70, 80, 90, 5000]).reshape(-1, 1)

# 2. Initialize the KBinsDiscretizer Algorithms
# encode='ordinal' returns the integer bin index (0, 1, 2, or 3) rather than a One-Hot encoded matrix
est_uniform = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='uniform')
est_quantile = KBinsDiscretizer(n_bins=4, encode='ordinal', strategy='quantile')

# 3. Fit the algorithms and transform the data
uniform_bins = est_uniform.fit_transform(income).flatten()
quantile_bins = est_quantile.fit_transform(income).flatten()

# 4. Visualization Architecture
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Plot 1: The Flawed Uniform Strategy
ax1.hist(uniform_bins, bins=np.arange(5)-0.5, edgecolor='black', color='tomato', alpha=0.8, rwidth=0.8)
ax1.set_title("Uniform Binning\n(Variance Destroyed)", fontsize=14, fontweight='bold')
ax1.set_xlabel("Bin Index", fontsize=12)
ax1.set_ylabel("Number of Individuals", fontsize=12)
ax1.set_xticks([0, 1, 2, 3])
ax1.set_ylim(0, 10)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Plot 2: The Rigorous Quantile Strategy
ax2.hist(quantile_bins, bins=np.arange(5)-0.5, edgecolor='black', color='mediumseagreen', alpha=0.8, rwidth=0.8)
ax2.set_title("Quantile Binning\n(Variance Preserved)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Bin Index", fontsize=12)
ax2.set_ylabel("Number of Individuals", fontsize=12)
ax2.set_xticks([0, 1, 2, 3])
ax2.set_ylim(0, 10)
ax2.grid(axis='y', linestyle='--', alpha=0.7)

plt.suptitle("Impact of Binning Strategies on Highly Skewed Data", fontsize=16, y=1.05)
plt.tight_layout()
plt.savefig('binning_comparison.png', bbox_inches='tight')

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

gold_file = Path('../data/gold_hist.txt')

with gold_file.open() as file:
    lines = [line.strip() for line in file if line.strip()]

years = [int(value) for value in lines[::2]]
prices = [float(value.replace('$', '').replace(',', '')) for value in lines[1::2]]

gold_df = pd.DataFrame({'year': years, 'price': prices})
gold_df.head()

In [ ]:
gold_df = gold_df.sort_values('year')
gold_df.info()

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gold_df['year'], gold_df['price'], color='goldenrod')
plt.title('Historical Gold Prices')
plt.xlabel('Year')
plt.ylabel('Gold Price (USD)')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
gold_since_1972 = gold_df.loc[gold_df['year'] >= 1972].copy()
peak_index = gold_since_1972['price'].idxmax()
peak_year = gold_since_1972.loc[peak_index, 'year']
peak_price = gold_since_1972.loc[peak_index, 'price']

peak_year, peak_price

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(gold_since_1972['year'], gold_since_1972['price'], color='steelblue')
plt.scatter(peak_year, peak_price, color='crimson', marker='^', s=120)
plt.annotate(
    f'Peak price: ${peak_price:,.2f} ({peak_year})',
    xy=(peak_year, peak_price),
    xytext=(peak_year - 8, peak_price - 150),
    arrowprops=dict(arrowstyle='->', color='black')
)
plt.title('Historical Gold Prices Since 1972')
plt.xlabel('Year')
plt.ylabel('Gold Price (USD)')
plt.grid(alpha=0.3)
plt.show()